## Complete Reviews Preproseccing (Cleaning + Normalization)

### Import Required Libraries

In [2]:
import pandas as pd
import re
import unicodedata
import string
from nltk.stem import WordNetLemmatizer
from textblob import TextBlob
import nltk
from nltk.corpus import stopwords

nltk.download('omw-1.4')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
df = pd.read_csv('../../data/raw/whole_dataset.csv')
df.head()

,review,language,label
0,terrible,en,negative
1,less movie show total money worst,en,negative
2,inapt brightness cannot brought zero even phon...,en,negative
3,able login account tried uninstalling reinstat...,en,negative
4,netflix one favourite out platform watch show ...,en,negative


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21984 entries, 0 to 21983
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   review    21981 non-null  object
 1   language  21984 non-null  object
 2   label     21984 non-null  object
dtypes: object(3)
memory usage: 515.4+ KB


In [5]:
# Verify the columns in the DataFrame
print("Columns after dropping author_rating:", df.columns.tolist())
print("Shape:", df.shape)

Columns after dropping author_rating: ['review', 'language', 'label']
Shape: (21984, 3)


In [6]:
# Check unique languages in the dataset
print("Unique languages before filtering:")
print(df['language'].value_counts())
print("\nTotal rows before filtering:", len(df))

Unique languages before filtering:
language
ar       14620
en        7323
emoji       41
Name: count, dtype: int64

Total rows before filtering: 21984


In [7]:
# Filter dataset to keep only 'ar', 'en', and 'emoji' languages
languages_to_keep = ['ar', 'en', 'emoji']
df = df[df['language'].isin(languages_to_keep)]

# Reset index after filtering
df = df.reset_index(drop=True)

print("Unique languages after filtering:")
print(df['language'].value_counts())
print("\nTotal rows after filtering:", len(df))


Unique languages after filtering:
language
ar       14620
en        7323
emoji       41
Name: count, dtype: int64

Total rows after filtering: 21984


In [8]:
# function to check if there is duplicate text in the dataset if there is it will return the rows with duplicate text otherwise print "No duplicates found"
def check_duplicates(df):
    duplicates = df[df.duplicated(subset=['review'], keep=False)]
    if not duplicates.empty:
        print("Duplicate reviews found:")
        return duplicates
    else:
        print("No duplicates found")
        return None
duplicates = check_duplicates(df)
# save the duplicates to a csv file with UTF-8 encoding
#duplicates.to_csv('duplicates.csv', index=False, encoding='utf-8')
display(duplicates.head())
print("Total duplicate rows:", len(duplicates))

Duplicate reviews found:


,review,language,label
12,bad,en,negative
26,okay,en,neutral
34,worst pp,en,negative
52,bad pp,en,negative
63,useless,en,negative


Total duplicate rows: 8734


In [9]:
def remove_duplicates(df):
    initial_count = len(df)
    df = df.drop_duplicates(subset=['review'])
    final_count = len(df)
    print(f"Removed {initial_count - final_count} duplicate rows.")
    return df

df = remove_duplicates(df)

Removed 4470 duplicate rows.


In [10]:
# Check for missing values in the arabic dataset
print("Missing values in the Arabic dataset:")
print(df.isnull().sum())

Missing values in the Arabic dataset:
review      1
language    0
label       0
dtype: int64


In [11]:
# remove rows with missing text
df = df.dropna(subset=['review'])
df.isnull().sum()

review      0
language    0
label       0
dtype: int64

### Cleaning & Normalizing Emoji's

In [12]:
#prints emoji rows

emoji_pattern = re.compile(
    r'^[\s\.\,\!\?\>\<\-\–\—•·➤➔❤♥️❣️💔💖💗💘💝💞💟🖤🤍🤎]*'
    r'([\U0001F1E0-\U0001F6FF'
    r'\U0001F900-\U0001F9FF'
    r'\U0001FA70-\U0001FAFF'
    r'\U00002702-\U000027B0'
    r'\U000024C2-\U0001F251'
    r'\u2600-\u26FF\u2700-\u27BF]+[\s\.\,\!\?\>\<\-\–\—•·]*)+$',
    flags=re.UNICODE
)

mask = df["review"].apply(
    lambda x: isinstance(x, str) and emoji_pattern.fullmatch(x.strip()) is not None
)

emoji_rows = df[mask.fillna(False)].copy()

print("Emoji-only rows found:", len(emoji_rows))
emoji_rows.head()

Emoji-only rows found: 78


,review,language,label
530,🤗,en,neutral
531,👍👍,en,neutral
532,👍😋,en,neutral
534,👌🏼👌🏼,en,neutral
549,👍 👌 👍 👌,en,neutral


In [13]:
#prints emoji symbol combos

emoji_only_pattern = re.compile(
    r'^[\s\.\,\!\?\>\<\-\–\—•·➤➔❤♥️❣️💔💖💗💘💝💞💟🖤🤍🤎'
    r'\U0001F1E0-\U0001F6FF'
    r'\U0001F300-\U0001FAFF'
    r'\U00002600-\U000026FF'
    r'\U00002700-\U000027BF]+$',
    flags=re.UNICODE
)

emoji_only_mask = df["review"].apply(
    lambda x: isinstance(x, str) and emoji_only_pattern.fullmatch(x.strip()) is not None
).fillna(False)

emoji_only_rows = df[emoji_only_mask].copy()

text_detector = re.compile(r'[A-Za-z\u0600-\u06FF]')
emoji_pattern = re.compile(r'[\U0001F300-\U0001FAFF\u2600-\u26FF\u2700-\u27BF]', flags=re.UNICODE)

def is_emoji_symbol_combo_v2(text):
    if not isinstance(text, str):
        return False

    has_emoji = any(emoji_pattern.search(char) for char in text)
    has_symbol = any(
        unicodedata.category(char).startswith(('P', 'S'))
        for char in text
    )
    has_text = bool(text_detector.search(text))

    return has_emoji and has_symbol and not has_text

emoji_only_pattern = re.compile(
    r'^[\s\.\,\!\?\>\<\-\–\—•·➤➔❤♥️❣️💔💖💗💘💝💞💟🖤🤍🤎'
    r'\U0001F1E0-\U0001F6FF'
    r'\U0001F300-\U0001FAFF'
    r'\U00002600-\U000026FF'
    r'\U00002700-\U000027BF]+$',
    flags=re.UNICODE
)

emoji_only_mask = df["review"].apply(
    lambda x: isinstance(x, str) and emoji_only_pattern.fullmatch(x.strip()) is not None
).fillna(False)

emoji_only_rows = df[emoji_only_mask].copy()

text_detector = re.compile(r'[A-Za-z\u0600-\u06FF]')
emoji_pattern = re.compile(r'[\U0001F300-\U0001FAFF\u2600-\u26FF\u2700-\u27BF]', flags=re.UNICODE)

def is_emoji_symbol_combo(text):
    if not isinstance(text, str):
        return False

    has_emoji = any(emoji_pattern.search(char) for char in text)
    has_symbol = any(
        unicodedata.category(char).startswith(('P', 'S'))
        for char in text
    )
    has_text = bool(text_detector.search(text))

    return has_emoji and has_symbol and not has_text

combo_mask = df["review"].apply(is_emoji_symbol_combo).fillna(False)
combo_rows = df[combo_mask].copy()

emoji_symbol_combo_rows = combo_rows.drop(index=emoji_only_rows.index, errors="ignore")

print("Emoji + symbol combo rows found:", len(emoji_symbol_combo_rows))
emoji_symbol_combo_rows.head()
combo_rows = df[combo_mask].copy()

emoji_symbol_combo_rows = combo_rows.drop(index=emoji_only_rows.index, errors="ignore")
emoji_symbol_combo_rows.head()

Emoji + symbol combo rows found: 3


,review,language,label
17704,910👌,en,positive
21979,9/10👌,en,positive
21980,100% 🤩,en,positive


In [14]:
import unicodedata

# Basic skin tone & variation selector Unicode ranges
SKIN_TONES = {
    '\U0001F3FB', '\U0001F3FC', '\U0001F3FD', '\U0001F3FE', '\U0001F3FF'
}
VARIATION_SELECTORS = {'\uFE0F', '\uFE0E'}

def normalize_emoji_string(text):
    if not isinstance(text, str):
        return ""
    
    # Remove skin tones and variation selectors
    cleaned = ''.join(
        ch for ch in text
        if ch not in SKIN_TONES and ch not in VARIATION_SELECTORS
    )
    
    # Filter only emoji-related characters
    emoji_chars = [ch for ch in cleaned if unicodedata.category(ch).startswith(('So', 'Sk'))]
    
    # Deduplicate and sort for semantic comparison
    normalized = ''.join(sorted(set(emoji_chars)))
    return normalized

# Create normalization column
emoji_only_rows["normalized"] = emoji_only_rows["review"].apply(normalize_emoji_string)

# Drop duplicates based on normalized form
emoji_only_deduped = emoji_only_rows.drop_duplicates(subset="normalized")

# Compare before and after
removed_count = len(emoji_only_rows) - len(emoji_only_deduped)
print(f"Removed {removed_count} semantically duplicate emoji-only rows")
emoji_only_deduped[["review", "normalized"]].head(10)

Removed 36 semantically duplicate emoji-only rows


,review,normalized
530,🤗,🤗
531,👍👍,👍
532,👍😋,👍😋
534,👌🏼👌🏼,👌
549,👍 👌 👍 👌,👌👍
551,💕,💕
557,😀😘😘,😀😘
719,👍🏻👌🏻😋,👌👍😋
788,😊,😊
857,👍😘♥️,♥👍😘


In [ ]:
# Ensure emoji is installed
import emoji
import re
import unicodedata

emoji_pattern = re.compile(r'[\U0001F300-\U0001FAFF\u2600-\u26FF\u2700-\u27BF]', flags=re.UNICODE)

def normalize_emojis(text):
    if not isinstance(text, str):
        return text

    result = []
    seen = set()

    for char in text:
        # Skip skin tone modifiers and variation selectors
        if char in SKIN_TONES or char in VARIATION_SELECTORS:
            continue

        # Check if char is an emoji using regex
        if emoji_pattern.match(char):
            # Demojize and remove skin tone descriptors if possible
            try:
                base = emoji.demojize(char)
                base = re.sub(r"_medium_skin_tone|_light_skin_tone|_dark_skin_tone|_medium-dark_skin_tone|_medium-light_skin_tone", "", base)
                unified = emoji.emojize(base, language='alias')
            except Exception:
                unified = char

            # Remove duplicates
            if unified not in seen:
                result.append(unified)
                seen.add(unified)
        else:
            result.append(char)

    return ''.join(result)

mask = df['review'].apply(lambda x: isinstance(x, str) and emoji_pattern.search(x) is not None)
df.loc[mask, 'review'] = df.loc[mask, 'review'].apply(normalize_emojis)

print(f"Re-normalized {mask.sum()} rows")

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Re-normalized 2503 rows
Re-normalized 2503 rows


### Normalizing Arabic Reviews

In [ ]:
import re

# 1. Compile patterns once
DIACRITICS_RE = re.compile(r'[\u0610-\u061A\u064B-\u0652\u06D6-\u06ED]')
TATWEEL_RE    = re.compile(r'[\u0640]')
NON_ARABIC_RE = re.compile(r'[^\u0600-\u06FF\s]')
ELONG_RE      = re.compile(r'(.)\1{2,}', re.UNICODE)

# 2. Character mapping for alef/hamza/ta-marbuta/ya
CHAR_MAP = str.maketrans({
    'آ':'ا','أ':'ا','إ':'ا',
    'ى':'ي','ؤ':'و','ئ':'ي','ة':'ه'
})

# 3. Clitic prefixes & suffixes
CLITIC_PREFS = ('ال','و','ف','ب','ك','ل','لل')
CLITIC_SUF  = ('ه','ها','هم','نا','كم','كن','ي')

def normalize_arabic(text, 
                     strip_diacritics=True,
                     strip_tatweel=True,
                     normalize_chars=True,
                     remove_non_arabic=True,
                     collapse_elongation=True,
                     split_clitics=True):
    """Run a full normalization pipeline on Arabic text."""
    text = text.strip()

    if strip_diacritics:
        text = DIACRITICS_RE.sub('', text)
    if strip_tatweel:
        text = TATWEEL_RE.sub('', text)
    if normalize_chars:
        text = text.translate(CHAR_MAP)
    if remove_non_arabic:
        text = NON_ARABIC_RE.sub(' ', text)
    if collapse_elongation:
        text = ELONG_RE.sub(r'\1\1', text)

    if split_clitics:
        tokens = []
        for w in text.split():
            # detach prefix
            pref = next((p for p in CLITIC_PREFS if w.startswith(p)), '')
            if pref:
                w = w[len(pref):]
                tokens.append(pref)
            # detach suffix
            suf = next((s for s in CLITIC_SUF if w.endswith(s)), '')
            if suf:
                w = w[:-len(suf)]
            tokens.append(w)
            if suf:
                tokens.append(suf)
        text = ' '.join(tokens)

    # final cleanup: collapse multiple spaces
    return re.sub(r'\s+', ' ', text).strip()
    



In [18]:
df.loc[df['language'] == 'ar', 'review'] = df.loc[df['language'] == 'ar', 'review'].apply(lambda review: normalize_arabic(str(review)))

In [19]:
display(df.head())

,review,language,label
0,terrible,en,negative
1,less movie show total money worst,en,negative
2,inapt brightness cannot brought zero even phon...,en,negative
3,able login account tried uninstalling reinstat...,en,negative
4,netflix one favourite out platform watch show ...,en,negative


In [20]:
df.info()
display(df['label'].unique())

<class 'pandas.core.frame.DataFrame'>
Index: 17513 entries, 0 to 21980
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   review    17513 non-null  object
 1   language  17513 non-null  object
 2   label     17513 non-null  object
dtypes: object(3)
memory usage: 547.3+ KB


array(['negative', 'neutral', 'positive'], dtype=object)

# Normalizing English Reviews

In [21]:
# Remove all newline and carriage return characters for English reviews only
df.loc[df['language'] == 'en', 'review'] = df.loc[df['language'] == 'en', 'review'].str.replace(r'[\r\n]+', ' ', regex=True)
# remove extra spaces again for English reviews only
df.loc[df['language'] == 'en', 'review'] = df.loc[df['language'] == 'en', 'review'].str.strip()
df.loc[df['language'] == 'en', 'review'] = df.loc[df['language'] == 'en', 'review'].str.replace(r'\s+', ' ', regex=True)
df.head()

,review,language,label
0,terrible,en,negative
1,less movie show total money worst,en,negative
2,inapt brightness cannot brought zero even phon...,en,negative
3,able login account tried uninstalling reinstat...,en,negative
4,netflix one favourite out platform watch show ...,en,negative


In [22]:
# Remove punctuation and symbols from 'review' column for English reviews only
df.loc[df['language'] == 'en', 'review'] = df.loc[df['language'] == 'en', 'review'].apply(
    lambda x: re.sub(f"[{re.escape(string.punctuation)}]", "", str(x))
)
df.loc[df['language'] == 'en', 'review'] = df.loc[df['language'] == 'en', 'review'].str.replace(r'\s+', ' ', regex=True).str.strip()
# Only apply the replacement to English reviews
df.loc[df['language'] == 'en', 'review'] = df.loc[df['language'] == 'en', 'review'].str.replace(r'["\']', "", regex=True)
df.head()

,review,language,label
0,terrible,en,negative
1,less movie show total money worst,en,negative
2,inapt brightness cannot brought zero even phon...,en,negative
3,able login account tried uninstalling reinstat...,en,negative
4,netflix one favourite out platform watch show ...,en,negative


In [23]:
# Convert text in 'review' column to lowercase for English reviews only
df.loc[df['language'] == 'en', 'review'] = df.loc[df['language'] == 'en', 'review'].str.lower()
df.head()

,review,language,label
0,terrible,en,negative
1,less movie show total money worst,en,negative
2,inapt brightness cannot brought zero even phon...,en,negative
3,able login account tried uninstalling reinstat...,en,negative
4,netflix one favourite out platform watch show ...,en,negative


In [24]:
# Download stopwords if not already
nltk.download('stopwords')

# Get English stopwords
stop_words = set(stopwords.words('english'))

# Remove stopwords for English reviews only
df.loc[df['language'] == 'en', 'review'] = df.loc[df['language'] == 'en', 'review'].apply(
    lambda x: ' '.join([word for word in str(x).split() if word not in stop_words])
)
df.head()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,review,language,label
0,terrible,en,negative
1,less movie show total money worst,en,negative
2,inapt brightness cannot brought zero even phon...,en,negative
3,able login account tried uninstalling reinstat...,en,negative
4,netflix one favourite platform watch show prob...,en,negative


In [25]:
def clean_noise(text):
    if pd.isnull(text):
        return ""
    
    # Remove repeated punctuation like !!, ??, ..., ----
    text = re.sub(r'([!?.\-])\1+', r'\1', text)

    # Replace multiple dots or punctuation with a space
    text = re.sub(r'[.]{2,}', ' ', text)

    # Remove multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Apply noise cleaning only to English reviews
df.loc[df['language'] == 'en', 'review'] = df.loc[df['language'] == 'en', 'review'].apply(clean_noise)
display(df)

,review,language,label
0,terrible,en,negative
1,less movie show total money worst,en,negative
2,inapt brightness cannot brought zero even phon...,en,negative
3,able login account tried uninstalling reinstat...,en,negative
4,netflix one favourite platform watch show prob...,en,negative
...,...,...,...
21976,👍,emoji,positive
21977,👌,emoji,positive
21978,🍟👌,emoji,positive
21979,910👌,en,positive


In [26]:
nltk.download('wordnet')
nltk.download('omw-1.4')

lemmatizer = WordNetLemmatizer()

def apply_lemmatization(text):
    if pd.isnull(text):
        return ""
    return ' '.join([lemmatizer.lemmatize(word) for word in text.split()])

# Apply lemmatization only to English reviews
df.loc[df['language'] == 'en', 'review'] = df.loc[df['language'] == 'en', 'review'].apply(apply_lemmatization)
display(df)

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


,review,language,label
0,terrible,en,negative
1,less movie show total money worst,en,negative
2,inapt brightness cannot brought zero even phon...,en,negative
3,able login account tried uninstalling reinstat...,en,negative
4,netflix one favourite platform watch show prob...,en,negative
...,...,...,...
21976,👍,emoji,positive
21977,👌,emoji,positive
21978,🍟👌,emoji,positive
21979,910👌,en,positive


In [27]:
import re

def reduce_repeated_chars(text):
    if pd.isnull(text):
        return ""
    # Replace 3 or more repeated characters with just 2
    return re.sub(r'(.)\1{2,}', r'\1\1', text)

# Reduce repeated characters only for English reviews
df.loc[df['language'] == 'en', 'review'] = df.loc[df['language'] == 'en', 'review'].apply(reduce_repeated_chars)

display(df)


,review,language,label
0,terrible,en,negative
1,less movie show total money worst,en,negative
2,inapt brightness cannot brought zero even phon...,en,negative
3,able login account tried uninstalling reinstat...,en,negative
4,netflix one favourite platform watch show prob...,en,negative
...,...,...,...
21976,👍,emoji,positive
21977,👌,emoji,positive
21978,🍟👌,emoji,positive
21979,910👌,en,positive


In [28]:
def correct_spelling(text):
    if pd.isnull(text):
        return ""
    corrected = TextBlob(text).correct()
    return str(corrected)

# Correct spelling only for English reviews
df.loc[df['language'] == 'en', 'review'] = df.loc[df['language'] == 'en', 'review'].apply(correct_spelling)
display(df)

,review,language,label
0,terrible,en,negative
1,less movie show total money worst,en,negative
2,inapt brightness cannot brought zero even phon...,en,negative
3,able login account tried uninstalling reinstat...,en,negative
4,netflix one favourite platform watch show prob...,en,negative
...,...,...,...
21976,👍,emoji,positive
21977,👌,emoji,positive
21978,🍟👌,emoji,positive
21979,910👌,en,positive


### Double Check if There are Nulls After normalizing

In [29]:
print("current rows:", len(df))
df = df.drop_duplicates(subset='review')
df = df.reset_index(drop=True)

print("Remaining rows:", len(df))

display(df)

current rows: 17513
Remaining rows: 17119


,review,language,label
0,terrible,en,negative
1,less movie show total money worst,en,negative
2,inapt brightness cannot brought zero even phon...,en,negative
3,able login account tried uninstalling reinstat...,en,negative
4,netflix one favourite platform watch show prob...,en,negative
...,...,...,...
17114,😍❤,emoji,positive
17115,🥰😍,emoji,positive
17116,😘😍,emoji,positive
17117,🍟👌,emoji,positive


In [30]:
print(df.isnull().sum())
df = df.dropna()
df = df.dropna(subset=['review'])
df = df.reset_index(drop=True)

print(df.isnull().sum())

display(df)

review      0
language    0
label       0
dtype: int64
review      0
language    0
label       0
dtype: int64


,review,language,label
0,terrible,en,negative
1,less movie show total money worst,en,negative
2,inapt brightness cannot brought zero even phon...,en,negative
3,able login account tried uninstalling reinstat...,en,negative
4,netflix one favourite platform watch show prob...,en,negative
...,...,...,...
17114,😍❤,emoji,positive
17115,🥰😍,emoji,positive
17116,😘😍,emoji,positive
17117,🍟👌,emoji,positive


In [ ]:
df.to_csv('../../data/complete2_dataset.csv', encoding='utf-8-sig', index=False)

In [ ]:
df = pd.read_csv('../../data/complete2_dataset.csv')
# make the neutral in both languages more and close to the number of positive and negative (sampling)
neutral_count = df[df['label'] == 'neutral'].shape[0]
positive_count = df[df['label'] == 'positive'].shape[0]
negative_count = df[df['label'] == 'negative'].shape[0]
print(f"Neutral count: {neutral_count}, Positive count: {positive_count}, Negative count: {negative_count}")
# Display the distribution of labels
label_counts = df['label'].value_counts()
print("Label distribution:")
print(label_counts)

Neutral count: 3510, Positive count: 7368, Negative count: 6241
Label distribution:
label
positive    7368
negative    6241
neutral     3510
Name: count, dtype: int64


In [ ]:
# resample and balance the dataset
from sklearn.utils import resample
# do real resampling
def resample_dataset(df, target_label, target_count):
    current_count = df[df['label'] == target_label].shape[0]
    if current_count < target_count:
        # Upsample
        df_target = df[df['label'] == target_label]
        df_upsampled = resample(df_target, 
                                replace=True, 
                                n_samples=target_count - current_count, 
                                random_state=42)
        return pd.concat([df, df_upsampled], ignore_index=True)
    elif current_count > target_count:
        # Downsample
        df_target = df[df['label'] == target_label]
        df_downsampled = resample(df_target, 
                                  replace=False, 
                                  n_samples=target_count, 
                                  random_state=42)
        return pd.concat([df[df['label'] != target_label], df_downsampled], ignore_index=True)
    else:
        return df
    
resampled_df = resample_dataset(df, 'neutral', positive_count)
resampled_df = resample_dataset(resampled_df, 'negative', positive_count)
resampled_df = resample_dataset(resampled_df, 'positive', positive_count)
# Display the new distribution
new_label_counts = resampled_df['label'].value_counts()
print("New label distribution after resampling:")
print(new_label_counts)
# Save the resampled DataFrame
resampled_df.to_csv('../../data/raw/balanced_dataset.csv', encoding='utf-8-sig', index=False)


New label distribution after resampling:
label
negative    7368
neutral     7368
positive    7368
Name: count, dtype: int64


In [34]:
display(resampled_df)

,review,language,label
0,terrible,en,negative
1,less movie show total money worst,en,negative
2,inapt brightness cannot brought zero even phon...,en,negative
3,able login account tried uninstalling reinstat...,en,negative
4,netflix one favourite platform watch show prob...,en,negative
...,...,...,...
22099,ل لاسف تغير طعم ه ك ثير ب اخر ال اشهر,ar,negative
22100,ل ا توجد مصداقي ه ف العروض مع ال علم ان ها معر...,ar,negative
22101,اقسم ب الل ه سيء جدا جدا هذا ال فرع مهما اعطيت...,ar,negative
22102,ال خدم ه ب ط ي جدا ال اكل ماعلي ه زين احل ي ش ...,ar,negative


In [37]:
resampled_df['language'].unique()

array(['en', 'ar', 'emoji'], dtype=object)